# 01 · Herfindahl–Hirschman Index (HHI)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os, warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROJECT_DIR = r'C:\Users\name\Documents\Bachelor Thesis\Empirical Analysis'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

# Original-today counterfactual: pre-reform universe + pre-reform FFW + 2025-26 data
orig = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_original_today.parquet'))
# Pro-forma Next-Gen TOPIX: post-reform criteria + Adjusted FFW + 2025-26 data
pf   = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_proforma.parquet'))

# FF_MktCap is NaN for quarantined rows (delisted / missing FFW / low coverage),
# so a single `.notna()` filter reliably yields the investable set in both files.
orig = orig[orig['FF_MktCap'].notna()].copy()
pf   = pf[pf['FF_MktCap'].notna()].copy()
orig['w'] = orig['FF_MktCap'] / orig['FF_MktCap'].sum()
pf['w']   = pf['FF_MktCap']  / pf['FF_MktCap'].sum()

print(f'Original-today : {len(orig):,} stocks    Pro-forma-today : {len(pf):,} stocks')


In [ ]:
# ── HHI ───────────────────────────────────────────────────────────────────────
# s_i = w_i * 100  (market shares as whole-number percentages)
orig['s2'] = (orig['w'] * 100) ** 2
pf['s2']   = (pf['w']   * 100) ** 2

hhi_orig  = orig['s2'].sum()
hhi_pf    = pf['s2'].sum()
delta_hhi = hhi_pf - hhi_orig

top10_w_orig = orig.nlargest(10, 'w')['w'].sum()
top10_w_pf   = pf.nlargest(10, 'w')['w'].sum()

print('=' * 48)
print(f'  HHI  Original TOPIX  : {hhi_orig:.4f}')
print(f'  HHI  Pro-forma TOPIX : {hhi_pf:.4f}')
print(f'  ΔHHI                 : {delta_hhi:+.4f}  ({delta_hhi/hhi_orig*100:+.2f}%)')
print(f'  Top-10 weight  Orig  : {top10_w_orig:.2%}')
print(f'  Top-10 weight  PF    : {top10_w_pf:.2%}')
print('=' * 48)

In [ ]:
# ── Statistical tests ─────────────────────────────────────────────────────────
# Welch t-test on squared weights
t_stat, p_ttest = stats.ttest_ind(pf['s2'].astype(float).dropna(), orig['s2'].astype(float).dropna(), equal_var=False)
print(f'Welch t-test on s²:  t = {t_stat:.4f}   p = {p_ttest:.4e}')

# Bootstrap 95% CI on ΔHHI
rng = np.random.default_rng(42)
boot = [
    rng.choice(pf['s2'].values,   size=len(pf),   replace=True).sum() -
    rng.choice(orig['s2'].values, size=len(orig), replace=True).sum()
    for _ in range(10_000)
]
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
print(f'Bootstrap 95% CI for ΔHHI: [{ci_lo:.4f}, {ci_hi:.4f}]')
sig = 'YES — significant' if (ci_lo > 0 or ci_hi < 0) else 'No — not significant'
print(f'  Statistically different from 0: {sig}')

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# A: HHI bar
ax = axes[0]
bars = ax.bar(['Original', 'Pro-forma'], [hhi_orig, hhi_pf],
              color=['steelblue', 'darkorange'], width=0.4, edgecolor='black')
ax.bar_label(bars, fmt='{:.4f}', padding=3)
ax.set_ylabel('HHI')
ax.set_title('A  HHI Concentration')
ax.annotate(f'ΔHHI = {delta_hhi:+.4f}\n({delta_hhi/hhi_orig*100:+.1f}%)',
            xy=(0.5, 0.6), xycoords='axes fraction', ha='center',
            bbox=dict(boxstyle='round', fc='lightyellow', ec='gray'))

# B: weight distribution
ax = axes[1]
ax.hist(np.log10(orig['w'].clip(1e-8)), bins=60, alpha=0.6, label='Original', color='steelblue')
ax.hist(np.log10(pf['w'].clip(1e-8)),   bins=60, alpha=0.6, label='Pro-forma', color='darkorange')
ax.set_xlabel('log₁₀(weight)')
ax.set_ylabel('Number of stocks')
ax.set_title('B  Weight Distribution')
ax.legend()

# C: Lorenz curve
ax = axes[2]
for label, df_p, color in [('Original', orig, 'steelblue'), ('Pro-forma', pf, 'darkorange')]:
    ws = np.sort(df_p['w'].values)
    ax.plot(np.arange(1, len(ws)+1)/len(ws), np.cumsum(ws)/ws.sum(),
            label=label, color=color)
ax.plot([0,1],[0,1],'k--', lw=0.8, label='Perfect equality')
ax.set_xlabel('Cumulative share of stocks')
ax.set_ylabel('Cumulative share of FF mktcap')
ax.set_title('C  Lorenz Curve')
ax.legend()

plt.suptitle('TOPIX Reform — HHI Concentration Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('hhi_analysis.png', bbox_inches='tight')
plt.show()
print('Saved: hhi_analysis.png')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Metric': ['N stocks', 'HHI', 'ΔHHI', 'ΔHHI (%)',
               'Top-10 weight', 'Max weight', 'Median weight',
               'Bootstrap 95% CI lower', 'Bootstrap 95% CI upper',
               'Welch t p-value'],
    'Original': [f'{len(orig):,}', f'{hhi_orig:.4f}', '', '',
                 f'{top10_w_orig:.2%}', f'{orig["w"].max():.3%}',
                 f'{orig["w"].median():.5%}', '', '', ''],
    'Pro-forma': [f'{len(pf):,}', f'{hhi_pf:.4f}',
                  f'{delta_hhi:+.4f}', f'{delta_hhi/hhi_orig*100:+.2f}%',
                  f'{top10_w_pf:.2%}', f'{pf["w"].max():.3%}',
                  f'{pf["w"].median():.5%}',
                  f'{ci_lo:.4f}', f'{ci_hi:.4f}', f'{p_ttest:.4e}']
})
print(summary.to_string(index=False))
summary.to_csv('hhi_summary.csv', index=False)
print('\nSaved: hhi_summary.csv')